In [1]:
import itertools
import logging
import os
from collections import Counter
from typing import Optional
from datetime import date, timedelta
import numpy as np
import pandas as pd
from deep_translator import GoogleTranslator

In [2]:
RAW_DATA_DIR    = r"Z:\Shared drives\External_COE_Hub\2. Workstream Files (by Function)\Claimback Processing\ID\Retail\Claimback Report Accrued\2026 05\Vinda\Data"

ADJUSTMENT_PATH = r"https://docs.google.com/spreadsheets/d/e/2PACX-1vTAjzdWEWisx4QENXgIvkQAKismoUI4satSm2WEBzmGClGoZOSsac-EBnlEI3Jf_YvgCx60zo2k90ll/pub?gid=253912155&single=true&output=csv"

SAVE_PATH       = r"C:\Users\thanh.bui\Documents\CLAIMBACK\ID\Retail\vinda"
os.makedirs(SAVE_PATH, exist_ok=True)

PERIOD_FILTER   = "05/31/2026"          # ← change per run

MASTER = r"https://docs.google.com/spreadsheets/d/e/2PACX-1vQsLdaxW91eSoesSd7eppin6U2w1UzWmVvRUyZPww2m0Ux9qWxpUVh4x7pcZrdeQ7YfWpmbY68vSEB1/pub?gid=450242995&single=true&output=csv"

"""
ACCOUNT BY BRAND (to get adjustment)
Amore SHP : 5885
Justmiss SHP: 6380
Vinda SHP: 6448
"""

ACCOUNT_NAME = "vinda"
ACCOUNT = "6448"

MASTER_PRICE = r"https://docs.google.com/spreadsheets/d/e/2PACX-1vT0cGuuufzYpued-fqmPJgqHP3U1M_cm7asl1lxp5DI2tpgLCEoL2YK5jY5D_3rGBOtmTNBb5KeGw3f/pub?gid=0&single=true&output=csv"

## Master price

In [3]:
mp = pd.read_csv(MASTER_PRICE)
mp['Brand'] = mp['Brand'].str.lower()
mp = mp[mp['Brand'] == ACCOUNT_NAME.lower()]
mp = mp.drop_duplicates(subset= "SKU", keep = 'last')
mp['Purchase Price (Including VAT)'] = (
    mp['Purchase Price (Including VAT)']
    .astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)
mp

,Date,Brand,SKU,SKU GP Rate,Consumer Price (Including VAT) / RRP,Purchase Price (Including VAT),Invoice Price (Including VAT)
0,5/31/2026,vinda,ID_HYB_VND-1703000700,15%,"18,919",16081.0,0
1,5/31/2026,vinda,ID_HYB_VND-1703000698,15%,"18,919",16081.0,0
2,5/31/2026,vinda,ID_HYB_VND-1703000699,15%,"18,919",16081.0,0
3,5/31/2026,vinda,ID_HYB_VND-1703000697,15%,"18,919",16081.0,0
4,5/31/2026,vinda,ID_HYB_VND-1700000710,15%,"31,532",26802.0,0
5,5/31/2026,vinda,ID_HYB_VND-1700000708,15%,"31,532",26802.0,0
6,5/31/2026,vinda,ID_HYB_VND-1700000711,15%,"31,532",26802.0,0
7,5/31/2026,vinda,ID_HYB_VND-1704000957,15%,"108,108",91892.0,0
8,5/31/2026,vinda,ID_HYB_VND-1708000229,15%,"66,667",56667.0,0
9,5/31/2026,vinda,ID_HYB_VND-1703000778,15%,"26,126",22207.0,0


## SHP

In [6]:
# ─────────────────────────────────────────────
# ACCOUNT CONFIGS
# ─────────────────────────────────────────────
"""
Config fields:
  label             – display name shown during processing
  output_name       – filename for the saved .xlsx result
  get_data          – keywords to match income files
  get_data_order    – keywords to match order files
  remove            – keywords that disqualify a file (use [] if none)
  account           – exact account number used to filter the adjustment CSV
  currency_cols     – column names for voucher / rebate (vary by market)
"""

# ─── logging ────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ─── known translation dictionaries ─────────────────────────────────────────
KNOWN_ADJ_TRANSLATIONS: dict[str, str] = {
    "Kompensasi atas pesanan hilang":                   "Compensation for Lost Orders",
    "Lainnya":                                          "Others",
    "Biaya Komisi AMS":                                 "AMS Commission Fee",
    "Penyesuaian Terkait Jasa Kirim":                   "Shipping Service Adjustment",
    "Pengembalian Barang/Dana setelah Dana Dilepaskan":  "Return of Goods/Funds after Funds Released",
}

KNOWN_STATUS_TRANSLATIONS: dict[str, str] = {
    # Return / Refund Status values
    "Pengembalian Dana":           "Refund",
    "Pengembalian Barang & Dana":  "Return & Refund",
    "Sedang Diproses":             "Processing",
    "Permintaan Diterima":         "Request Approved",
    "Dibatalkan":                  "Cancelled",
    "Selesai":                     "Completed",
    # Order Status values
    "Pesanan Selesai":             "Completed",
    "Dibatalkan":                  "Cancelled",
    "Sedang Dikirim":              "Shipped",
}

KNOWN_HEADER_TRANSLATIONS: dict[str, str] = {
    "No. Pesanan":                      "Order ID",
    "Status Pesanan":                   "Order Status",
    "Alasan Pembatalan":                "Cancel reason",
    "Status Pembatalan/ Pengembalian":  "Return / Refund Status",
    "No. Resi":                         "Tracking No.",
    "SKU Induk":                        "Parent SKU",
    "Nama Produk":                      "Product name",
    "Nomor Referensi SKU":              "SKU Reference No.",
    "Harga Awal":                       "Original Price",
    "Harga Setelah Diskon":             "Deal Price",
    "Jumlah":                           "Quantity",
    "Returned quantity":                "Returned Quantity",
    "Subtotal Pesanan":                 "Total Buyer Payment",
    "Diskon Dari Penjual":              "Seller Voucher",
    "Diskon Dari Shopee":               "Shopee Rebate",
}

ACCOUNTS: list[dict] = [
    {
        "label":          f"{ACCOUNT_NAME} SHP",
        "output_name":    f"{ACCOUNT_NAME}_SHP.xlsx",
        "get_data":       ["shp", "income"],
        "get_data_order": ["shp", "order"],
        "remove":         [],
        "account":        ACCOUNT,
        "currency_cols": {
            "voucher": "Seller Voucher",
            "rebate":  "Shopee Rebate",
        },
    },
]


# ═══════════════════════════════════════════════════════════════════
# SHARED HELPERS
# ═══════════════════════════════════════════════════════════════════

def clean_numeric(series: pd.Series) -> pd.Series:
    """
    Handle Indo format (dot = thousands separator):
      "404.600"   → 404600
      "2.010.000" → 2010000
      "404.60"    → 404.6  (decimal thật, giữ nguyên)
      "0"         → 0
    """
    def parse_one(val: str) -> str:
        val = str(val).strip()
        val = val.replace(".", "")   # remove tất cả dấu chấm (thousand separator)
        val = val.replace(",", ".")  # đổi dấu phẩy thành chấm (decimal separator)
        return val
    return pd.to_numeric(series.apply(parse_one), errors="coerce")

def batch_translate(series: pd.Series, known: dict[str, str]) -> pd.Series:
    """
    Translate a Series using `known` first; only call the API for
    values that are missing from the dictionary (batched in one call).
    The `known` dict is mutated in-place so subsequent calls are free.
    """
    unique_vals = series.dropna().unique()
    missing     = [v for v in unique_vals if v not in known and str(v).strip()]
    if missing:
        try:
            translated = GoogleTranslator(source="id", target="en").translate_batch(missing)
            known.update(dict(zip(missing, translated)))
        except Exception as exc:
            log.warning("Translation API failed for %d values: %s", len(missing), exc)
    return series.map(known).fillna(series)


def file_matches(filename: str, include: list[str], exclude: list[str]) -> bool:
    """Return True when filename (lowercased) contains ALL include keywords
    and NONE of the exclude keywords."""
    s = filename.lower()
    return (
        all(k in s for k in include)
        and not any(k in s for k in exclude)
        and "~$" not in filename
    )


# ═══════════════════════════════════════════════════════════════════
# STEP 1 – INCOME REPORT
# ═══════════════════════════════════════════════════════════════════

def get_income_report(
    income_list:     list[str],
    adjustment_path: str,
    get_data:        list[str],
    remove:          list[str],
    account:         str,
    period_filter:   str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load every income sheet that matches `get_data`/`remove`, then load
    and filter the shared adjustment CSV for the given account + period.

    Returns (df_income, df_adjust).  df_adjust is empty when nothing matches.
    """
    # ── income ──────────────────────────────────────────────────────
    df_dict: dict[str, pd.DataFrame] = {}
    for filename in income_list:
        if not file_matches(filename, get_data, remove):
            continue
        excel_file = pd.ExcelFile(filename)
        for sh in excel_file.sheet_names:
            if "income" in sh.lower():
                df_dict[sh] = pd.read_excel(
                    filename, sheet_name=sh, skiprows=5,
                    dtype={"Order ID": str, "Refund Amount": float},
                )
            if 'summary' in sh.lower():
                summary = pd.read_excel(filename, sheet_name = sh, skiprows = 10)
                summary.columns = ['fee', 'sub_fee', 'amount', 'total_amt']
                summary['fee'] = summary['fee'].ffill()
                                       
    if not df_dict:
        raise FileNotFoundError(
            f"No income file matched keywords {get_data} (excluding {remove}). "
            "Check that the raw-data folder contains the right files."
        )

    df_income = pd.concat(df_dict.values(), ignore_index=True)
    df_income["Order ID"] = df_income["Order ID"].astype(str).str.strip()

    # ── adjustment ──────────────────────────────────────────────────
    df_adjust = pd.read_csv(adjustment_path)

    df_adjust['Adjustment Amount'] = df_adjust["Adjustment Amount"].str.replace(",", "").astype(float)
    df_adjust['Refund Amount'] = df_adjust["Refund Amount"].str.replace(",", "").astype(float)
    
    df_adjust["Adjustment Type (EN)"] = batch_translate(
        df_adjust["Adjustment Type | Description"], KNOWN_ADJ_TRANSLATIONS
    )

    # Exact account match to avoid substring collisions (e.g. 5885 vs 15885)
    df_adjust = df_adjust[df_adjust["Customer"].str.contains(account, na=False)]
    df_adjust = df_adjust[df_adjust["Period Parent"] == period_filter]

    if df_adjust.empty:
        return df_income, pd.DataFrame()

    df_adjust = (
        df_adjust
        .groupby(["Adjustment Type | Description", "Adjustment Type (EN)", "Linked Order No."])[
            ["Adjustment Amount", "Refund Amount"]
        ]
        .sum()
        .reset_index()
    )

    return df_income, df_adjust, summary

# ═══════════════════════════════════════════════════════════════════
# STEP 2 – ORDER REPORT
# ═══════════════════════════════════════════════════════════════════

def get_order_report(
    order_list:     list[str],
    df_income:      pd.DataFrame,
    df_adjust:      pd.DataFrame,
    get_data_order: list[str],
    remove:         list[str],
    save_path:      str,
) -> tuple[pd.Series, pd.DataFrame, pd.DataFrame]:
    """
    Load order files, translate headers, then identify:
      - missing  : Order IDs in income (or adjustment) but not in orders
      - outlier  : Adjustment rows whose linked order is not in income
    Missing orders are also written to missing_orders.csv for reference.
    """
    order_frames: list[pd.DataFrame] = []
    for filename in order_list:
        if file_matches(filename, get_data_order, remove):
            order_frames.append(
                pd.read_excel(filename, dtype= str)
            )

    if not order_frames:
        raise FileNotFoundError(
            f"No order file matched keywords {get_data_order} (excluding {remove})."
        )

    df_order = pd.concat(order_frames, ignore_index=True)
    # Trong get_order_report, trước khi gọi batch_translate:
    df_order.columns = df_order.columns.str.strip().str.replace(r'\s+', ' ', regex=True)
    df_order.columns = batch_translate(
        pd.Series(df_order.columns), KNOWN_HEADER_TRANSLATIONS
    ).tolist()
    numeric_cols = [
    "Total Buyer Payment", "Deal Price", "Original Price", "Seller Discount", "Quantity", "Returned Quantity",
    "Seller Voucher", "Shopee Rebate",
    ]
    for col in numeric_cols:
        if col in df_order.columns:
            df_order[col] = clean_numeric(df_order[col])

    for col in ["Return / Refund Status", "Order Status"]:
        if col in df_order.columns:
            df_order[col] = batch_translate(df_order[col], KNOWN_STATUS_TRANSLATIONS)
    df_order["Order ID"] = df_order["Order ID"].astype(str).str.strip()

    income_ids = df_income["Order ID"].drop_duplicates()
    order_ids  = df_order["Order ID"].drop_duplicates()
    missing_orders = income_ids[~income_ids.isin(order_ids)]

    if df_adjust.empty:
        missing = missing_orders.drop_duplicates().reset_index(drop=True)
        outlier = pd.DataFrame()
    else:
        adjust         = df_adjust[~df_adjust["Linked Order No."].isin(order_ids)]
        missing_adjust = adjust["Linked Order No."]
        missing        = pd.concat([missing_orders, missing_adjust]).drop_duplicates().reset_index(drop=True)
        outlier        = df_adjust[~df_adjust["Linked Order No."].isin(income_ids)].reset_index()

    if not missing.empty:
        csv_path = os.path.join(save_path, "missing_orders.csv")
        missing.to_csv(csv_path, index=False)
        log.warning("%d missing order(s) — saved to %s", len(missing), csv_path)

    return missing, df_order, outlier


# ═══════════════════════════════════════════════════════════════════
# STEP 3 – COMPENSATION  (split into three focused functions)
# ═══════════════════════════════════════════════════════════════════

def _merge_orders_with_income(
    df_order:      pd.DataFrame,
    df_income:     pd.DataFrame,
    outlier:       pd.DataFrame,
    df_adjust:     pd.DataFrame,
    currency_cols: dict[str, str],
) -> pd.DataFrame:
    """Merge order rows with income and append any outlier orders."""
    df = pd.merge(
        df_order,
        df_income[["Order ID", "Voucher Sponsored by Seller"]],
        how="right",
        on="Order ID",
    )
    # Thêm log để biết bao nhiêu rows bị NaN sau merge:
    n_unmatched = df[df["SKU Reference No."].isna()]["Order ID"].nunique()
    if n_unmatched:
        log.warning("%d income order(s) có không có trong order file sau merge", n_unmatched)
    if not df_adjust.empty:
        linked_orders = outlier.get("Linked Order No.", pd.Series(dtype=object))
        df_outlier    = df_order[df_order["Order ID"].isin(linked_orders)].copy()
        df_outlier["Voucher Sponsored by Seller"] = 0
        df = pd.concat([df, df_outlier], ignore_index=True)

    voucher_col = currency_cols["voucher"]
    df["index"]            = df.groupby("Order ID").cumcount() + 1
    df["Voucher"]          = np.where(df["index"] == 1, -df["Voucher Sponsored by Seller"], 0)
    df["Quantity mapping"] = df["SKU Reference No."].str.extract(r'[xX](\d+)$')
    df["Quantity mapping"] = pd.to_numeric(df["Quantity mapping"], errors="coerce").fillna(1).astype(int)
    df['sku_mapping']      = np.where(df['SKU Reference No.'].str.contains('GIFT'), 
                                   df['SKU Reference No.'], 
                                   df['SKU Reference No.'].str.replace(r'[xX]\d+$', '', regex=True))
    df["total_quant"] = df["Quantity"] * df["Quantity mapping"]
    return df

def _classify_adjustments(df_adjust: pd.DataFrame) -> pd.DataFrame:
    """Add a 'classify' column: reimbursement / return / other."""
    if df_adjust.empty:
        return df_adjust
    desc = df_adjust["Adjustment Type (EN)"].str.lower()
    df_adjust = df_adjust.copy()
    df_adjust["classify"] = np.select(
        [
            desc.str.contains("logistic|other|lost|shipping|delivery", na=False, regex=True),
            desc.str.contains("return|refund",        na=False, regex=True),
        ],
        ["reimbursement", "return"],
        default="other",
    )
    df_adjust['Adjustment Amount'] = np.where(df_adjust['Refund Amount'].notna(), df_adjust['Refund Amount'], df_adjust['Adjustment Amount'])
    df_adjust = df_adjust.groupby(['Linked Order No.', 'classify'])['Adjustment Amount'].sum().reset_index()
    return df_adjust

def _compute_refund_totals(
    df_income: pd.DataFrame,
    df_adjust: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Merge in-month return adjustments into income's Refund Amount,
    then build a refund summary (only negative net amounts).
    Returns (df_income_updated, refund_summary).
    """
    
    if not df_adjust.empty:
        print(df_adjust.info())
        refund_list = df_adjust[df_adjust["classify"] == "return"]
        df_income   = df_income.merge(
            refund_list[["Linked Order No.", "Adjustment Amount"]],
            how="left",
            left_on="Order ID",
            right_on="Linked Order No.",
        )
        df_income["Refund Amount"] = (
            df_income["Refund Amount"] + df_income["Adjustment Amount"].fillna(0)
        )
        df_income = df_income.drop(
            columns=["Linked Order No.", "Adjustment Amount"], errors="ignore"
        )

    df_refund = (
        df_income.groupby("Order ID")["Refund Amount"]
        .sum()
        .rename("total_refund")
        .reset_index()
    )
    refund = df_refund[df_refund["total_refund"] < 0].copy()
    refund["total_refund"] = (refund["total_refund"] * -1).round(2)

    df_income = df_income.drop(columns=["Voucher Sponsored by Seller"], errors="ignore")
    return df_income, refund


def get_compensation(
    df_order:      pd.DataFrame,
    df_income:     pd.DataFrame,
    outlier:       pd.DataFrame,
    df_adjust:     pd.DataFrame,
    currency_cols: dict[str, str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Orchestrate the three sub-steps and return all outputs."""
    df_adjust   = _classify_adjustments(df_adjust)
    df_comp     = _merge_orders_with_income(df_order, df_income, outlier, df_adjust, currency_cols)
    df_income, refund = _compute_refund_totals(df_income, df_adjust)

    refund_orders = df_comp[df_comp["Order ID"].isin(refund["Order ID"])].reset_index(drop=True)
    return df_comp, refund, refund_orders, df_adjust, df_income


# ═══════════════════════════════════════════════════════════════════
# STEP 4 – REFUND DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════

def dp_refund_distribution(
    df_ref:        pd.DataFrame,
    refund_target: float,
    tol:           float = 0.01,
) -> Optional[dict[str, int]]:
    """
    Exact subset-sum via itertools.combinations — avoids recursion limits.
    Returns a {sku: qty} dict if an exact (within tol) match is found,
    else None.
    """
    items: list[tuple[str, float]] = []
    for _, row in df_ref.iterrows():
        for _ in range(int(row["total_quant"])):
            items.append((row["SKU Reference No."], row["unit_price"]))

    for size in range(1, len(items) + 1):
        for combo in itertools.combinations(items, size):
            if abs(sum(p for _, p in combo) - refund_target) <= tol:
                return dict(Counter(sku for sku, _ in combo))
    return None


def fallback_refund_distribution(
    df_ref:        pd.DataFrame,
    refund_target: float,
) -> tuple[dict[str, int], float]:
    """
    Greedy best-fit when no exact combo exists.
    Returns ({sku: qty}, approximate_total).
    """
    skus        = df_ref["SKU Reference No."].tolist()
    unit_prices = df_ref["unit_price"].to_numpy()
    max_qty     = df_ref["total_quant"].astype(int).to_numpy()
    n           = len(skus)
    refund_qty  = np.zeros(n, dtype=int)
    current_sum = 0.0

    while True:
        best_idx, best_gap = None, abs(refund_target - current_sum)
        for i in range(n):
            if refund_qty[i] >= max_qty[i]:
                continue
            gap = abs(refund_target - (current_sum + unit_prices[i]))
            if gap < best_gap:
                best_gap, best_idx = gap, i
        if best_idx is None:
            break
        refund_qty[best_idx] += 1
        current_sum += unit_prices[best_idx]

    return dict(zip(skus, refund_qty)), current_sum


def resolve_partial_refund(
    df_order_one:  pd.DataFrame,
    refund_target: float,
    tol:           float = 0.01,
) -> dict:
    cons   = (
        (df_order_one["Order Status"] == "Cancelled")
        | df_order_one["Return / Refund Status"].notna()
    )
    df_ref = df_order_one[cons].copy().reset_index(drop=True)

    if df_ref.empty or refund_target <= 0:
        return {"method": "NO_DATA", "breakdown": None, "refunded_amount": 0.0}

    df_ref["Total Buyer Payment"] = pd.to_numeric(df_ref["Total Buyer Payment"], errors="coerce").fillna(0)
    df_ref["total_quant"]         = pd.to_numeric(df_ref["total_quant"],         errors="coerce").fillna(1)

    df_ref["unit_price"] = df_ref["Total Buyer Payment"] / df_ref["total_quant"].replace(0, np.nan)

    dp_result = dp_refund_distribution(df_ref, refund_target, tol)
    if dp_result is not None:
        refunded_amount = sum(
            qty * df_ref.loc[df_ref["SKU Reference No."] == sku, "unit_price"].iloc[0]
            for sku, qty in dp_result.items()
        )
        return {"method": "DP_EXACT", "breakdown": dp_result, "refunded_amount": refunded_amount}

    fallback_result, approx_sum = fallback_refund_distribution(df_ref, refund_target)
    return {"method": "GREEDY_ESTIMATE", "breakdown": fallback_result, "refunded_amount": approx_sum}


def process_all_orders(
    df_order:  pd.DataFrame,
    df_income: pd.DataFrame,
) -> pd.DataFrame:
    """
    For every order that has a refund, resolve how that refund distributes
    across SKUs.  Uses an index on df_income for O(1) look-ups.
    """
    income_index = df_income.set_index("Order ID")
    results      = []
    df_order = df_order.reset_index(drop=True)

    for order_id, group in df_order.groupby("Order ID"):
        if order_id not in income_index.index:
            continue
        refund_target = float(income_index.at[order_id, "total_refund"])
        res           = resolve_partial_refund(group, refund_target)
        results.append(
            {
                "Order ID":         order_id,
                "Refund target":    refund_target,
                "Refund method":    res["method"],
                "Refund breakdown": res["breakdown"],
                "Refunded amount":  res["refunded_amount"],
                "Variance":         res["refunded_amount"] - refund_target,
            }
        )

    return pd.DataFrame(results)


# ═══════════════════════════════════════════════════════════════════
# STEP 5 – CORRECT COMPENSATION
# ═══════════════════════════════════════════════════════════════════

def _expand_refund_breakdown(result_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in result_df.iterrows():
        if not isinstance(r["Refund breakdown"], dict):
            continue
        for sku, qty in r["Refund breakdown"].items():
            rows.append(
                {
                    "Order ID":               r["Order ID"],
                    "SKU Reference No.":      sku,
                    "correct_refund_quantity": int(qty),
                }
            )
    return pd.DataFrame(rows)


def apply_correct_refund_quantity(
    df_order:  pd.DataFrame,
    result_df: pd.DataFrame,
) -> pd.DataFrame:
    refund_map = _expand_refund_breakdown(result_df)
    df_out     = df_order.merge(refund_map, on=["Order ID", "SKU Reference No."], how="left")
    df_out["correct_refund_quantity"] = df_out["correct_refund_quantity"].fillna(0).astype(int)
    return df_out

def correct_compensation(
    df_comp:       pd.DataFrame,
    df_append:     pd.DataFrame,
    df_adjust:     pd.DataFrame,
    df_income:     pd.DataFrame,
    refund:        pd.DataFrame,
    mp:            pd.DataFrame,
    currency_cols: dict[str, str],
) -> pd.DataFrame:
    rebate_col = currency_cols["rebate"]

    # ── base frame: non-refund orders ───────────────────────────────
    df_final = df_comp[~df_comp["Order ID"].isin(refund["Order ID"])].copy().reset_index(drop=True)
    df_final.loc[
        df_final["Return / Refund Status"] == "Request Approved",
        "Return / Refund Status",
    ] = np.nan
    df_final["correct_refund_quantity"] = 0
    df_final = pd.concat([df_final, df_append], ignore_index=True)

    # ── rebate ───────────────────────────────────────────────────────
    df_final["Rebate Shopee"] = df_final[rebate_col]
    df_final.loc[
        df_final["Return / Refund Status"] == "Request Approved", "Rebate Shopee"
    ] = 0
    df_final["Rebate Shopee"] = df_final["Rebate Shopee"].fillna(0)

    # ── selling quantity ─────────────────────────────────────────────
    df_final["Selling quantity"] = df_final["total_quant"] - df_final["correct_refund_quantity"]

    # ── refund per unit mapping ──────────────────────────────────────
    mapping = (
        df_final.groupby("Order ID")["correct_refund_quantity"]
        .sum()
        .reset_index()
    )
    mapping = mapping.merge(df_income[["Order ID", "Refund Amount"]], how="left", on="Order ID")
    mapping["refund_per_unit"] = np.where(
        mapping["correct_refund_quantity"] == 0,
        0,
        mapping["Refund Amount"] / mapping["correct_refund_quantity"],
    )

    # ── core financials ──────────────────────────────────────────────
    df_finals = df_final.merge(
        mapping[["Order ID", "refund_per_unit"]], how="left", on="Order ID"
    )
    # Guard against division by zero on total_quant
    safe_quant = df_finals["total_quant"].replace(0, np.nan)
    df_finals["Refund Amount"] = (
        df_finals['refund_per_unit'] * df_finals["correct_refund_quantity"]
    ).round(2).fillna(0)

    df_finals["Deal_price"] = df_finals["Total Buyer Payment"]
    df_finals["TNMV"]       = (
        df_finals["Deal_price"]
        + df_finals["Refund Amount"]
        + df_finals["Rebate Shopee"]
        - df_finals["Voucher"]
    )

    # ── adjustment classification ────────────────────────────────────
    if not df_adjust.empty:
        df_adjust     = df_adjust[df_adjust["classify"] != "other"]
        reim_list     = df_adjust[df_adjust["classify"] == "reimbursement"]
        refund_list   = df_adjust[df_adjust["classify"] == "return"]
        refunds_income = refund_list[
            refund_list["Linked Order No."].isin(df_income["Order ID"])
        ]["Linked Order No."]
        refund_list = refund_list[~refund_list["Linked Order No."].isin(df_income["Order ID"])]
        df_adjust     =  df_adjust[~df_adjust["Linked Order No."].isin(refunds_income)]
        # remove matching orders from previous_return list and reim_list to use only reim as final amount
        remove_orders = refund_list[refund_list["Linked Order No."].isin(reim_list["Linked Order No."])]["Linked Order No."]
        df_adjust = df_adjust[~df_adjust["Linked Order No."].isin(remove_orders)]
        refund_list = refund_list[~refund_list["Linked Order No."].isin(remove_orders)]
    else:
        reim_list      = pd.DataFrame(columns=["Linked Order No."])
        refunds_income = pd.Series(dtype=object)
        refund_list    = pd.DataFrame(columns=["Linked Order No."])

    # Reimbursed orders keep their full quantity
    df_finals.loc[
        df_finals["Order ID"].isin(reim_list["Linked Order No."]),
        "Selling quantity",
    ] = df_finals["total_quant"]

    # ── per-SKU adjustment amount ────────────────────────────────────
    to_map = (
        df_finals.groupby("Order ID")["Selling quantity"]
        .sum()
        .rename("total_sell")
        .reset_index()
    )
    
    if not df_adjust.empty:
        df_adjust = df_adjust.merge(
            to_map[["Order ID", "total_sell"]],
            how="left",
            left_on="Linked Order No.",
            right_on="Order ID",
        )
        df_adjust["adjust_per_SKU"] = (
            df_adjust["Adjustment Amount"] / df_adjust["total_sell"].replace(0, np.nan)
        )
        df_finals = df_finals.merge(
            df_adjust[["Order ID", "classify", "adjust_per_SKU"]],
            how="left",
            on="Order ID",
        )

    # ── adjustment month flag ────────────────────────────────────────
    df_finals["adjust_month"] = np.select(
        [
            df_finals["Order ID"].isin(refunds_income),
            df_finals["Order ID"].isin(refund_list["Linked Order No."]),
        ],
        ["in month", "previous month"],
        default="no adjustment",
    )
    # correct TNMV with order fully refunded
    non_selling = df_finals.groupby("Order ID")['Selling quantity'].sum().reset_index()
    non_selling = non_selling[non_selling['Selling quantity'] == 0]['Order ID']
    cons = df_finals['Order ID'].isin(non_selling)
    df_finals.loc[cons, 'TNMV'] = 0

    #calculate purchase price
    df_finals = df_finals.merge(mp[['SKU', 'Purchase Price (Including VAT)']] , 
                              how = 'left', left_on = "sku_mapping", right_on= "SKU")
    
    df_finals['Total Purchase price'] = df_finals['Selling quantity'] * df_finals['Purchase Price (Including VAT)'].fillna(0)
    
    if not df_adjust.empty:
        prev = df_finals["adjust_month"] == "previous month"
        reim = df_finals["Order ID"].isin(reim_list["Linked Order No."])
        zero_return = (df_finals["classify"] == "return") & (df_finals["Selling quantity"] == 0)
        df_finals.loc[zero_return, ["TNMV", "Voucher"]] = 0
        df_finals.loc[prev, "TNMV"] = (
            df_finals.loc[prev, "adjust_per_SKU"].fillna(0)
            * df_finals.loc[prev, "Selling quantity"]
        )
        df_finals.loc[prev, "Selling quantity"] = (-df_finals.loc[prev, 'correct_refund_quantity'])
        df_finals.loc[reim, "TNMV"] = df_finals.loc[reim, "adjust_per_SKU"].fillna(0) * df_finals.loc[reim, "Selling quantity"] 
        df_finals.loc[prev, 'Total Purchase price'] = df_finals.loc[prev, 'Total Purchase price'] * df_finals.loc[prev, "Selling quantity"]

    
    # df_finals = df_finals.drop(columns=["refund_per_unit", "adjust_per_SKU", "Voucher Sponsored by Seller", 
    #                                    "Quantity mapping", "Purchase Price (Including VAT)", 'SKU' ], errors="ignore")
    return df_finals
    
def format_df(df_final, inc, summary):
    expense = ['Fees & Charges', 'Shipping Subtotal']
    fees = summary[summary['fee'].isin(expense) & (summary['amount'] != 0)]['sub_fee']

    # filter cols in income to map with df_final
    cols = [c for c in fees if c in inc.columns]
    df_final = df_final.merge(inc[['Order ID'] + cols], how = 'left', on = 'Order ID')
    df_final[cols] = df_final[cols].fillna(0)
    
    df_final['rate'] = (df_final['total_quant'] / (df_final.groupby('Order ID')['total_quant'].transform('sum')))
    df_final[cols] = df_final[cols].mul(df_final['rate'], axis=0)

    # get only columns you want to present for clients
    df_final = df_final[['Order ID', 'Order Status', 'Time Order Created', 'SKU Reference No.', 'Time Payment is Made', 
                  'Payment Method', 'Product name',	'SKU Reference No.', 'Variation Name', 'sku_mapping',
                  'Voucher', 'total_quant', 'correct_refund_quantity', 'Rebate Shopee', 'Selling quantity', 'Refund Amount', 'Deal_price', 
                  'Total Purchase price', 'TNMV', 'rate'] + cols]    
    
    return df_final
    
# ═══════════════════════════════════════════════════════════════════
# STEP 6 – PIPELINE RUNNER
# ═══════════════════════════════════════════════════════════════════

def run_pipeline(
    config:                   dict,
    list_files:               list[str],
    adjustment_path:          str,
    save_path:                str,
    period_filter:            str,
    exclude_adjustment_types: Optional[list[str]] = None,
    auto_continue:            bool                = False,
) -> pd.DataFrame:
    """
    Run the full reconciliation pipeline for one account config.

    Parameters
    ----------
    config                   : one entry from ACCOUNTS
    list_files               : all raw file paths (income + order)
    adjustment_path          : path to the shared adjustment CSV
    save_path                : directory where outputs are written
    period_filter            : e.g. "04/30/2024"
    exclude_adjustment_types : adjustment type strings to drop (skips prompt)
    auto_continue            : if True, skip the missing-order pause
    """
    log.info("=" * 60)
    log.info("Processing: %s", config["label"])
    log.info("=" * 60)

    currency_cols = config.get("currency_cols", {
        "voucher": "Seller Voucher",
        "rebate":  "Shopee Rebate",
    })

    # ── Step 1 ───────────────────────────────────────────────────────
    income, adjusts, summary = get_income_report(
        list_files, adjustment_path,
        config["get_data"], config["remove"],
        config["account"], period_filter,
    )

    # ── Decide which adjustment types to exclude ─────────────────────
    if exclude_adjustment_types is not None:
        exclude = exclude_adjustment_types
    elif not adjusts.empty:
        log.info("Adjustment types found:\n%s", adjusts["Adjustment Type (EN)"].unique())
        raw     = input("Enter types to EXCLUDE (comma-separated), or press Enter to keep all: ").strip()
        exclude = [x.strip() for x in raw.split(",")] if raw else []
    else:
        exclude = []
        print("empty Adjustment")
    adjusted = (
        adjusts[~adjusts["Adjustment Type (EN)"].isin(exclude)]
        if "Adjustment Type (EN)" in adjusts.columns
        else adjusts
    )

    # ── Step 2 ───────────────────────────────────────────────────────
    missing, order, outliers = get_order_report(
        list_files, income, adjusted,
        config["get_data_order"], config["remove"],
        save_path,
    )

    if missing.empty:
        log.info("✓ No missing orders.")
    else:
        log.warning("%d missing order(s) detected.", len(missing))
        if not auto_continue:
            input("Press Enter to continue (or download older months first if orders are missing): ")

    # ── Steps 3 – 5 ──────────────────────────────────────────────────
    comp, refund, refund_orders, adj, income1 = get_compensation(
        order, income, outliers, adjusted, currency_cols
    )
    df_return  = process_all_orders(refund_orders, refund)
    df_append  = apply_correct_refund_quantity(refund_orders, df_return)
    final_df   = correct_compensation(comp, df_append, adj, income1, refund, mp, currency_cols)
    final      = format_df(final_df, income, summary)
                           
    # ── Save ─────────────────────────────────────────────────────────
    os.makedirs(save_path, exist_ok=True)
    output_path = os.path.join(save_path, config["output_name"])
    final.to_excel(output_path, index=False)
    log.info("✓ Saved → %s", output_path)
    return final


In [7]:

# ═══════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import glob

    list_files = glob.glob(os.path.join(RAW_DATA_DIR, "**", "*.xlsx"), recursive=True)
    
    for account_config in ACCOUNTS:
        run_pipeline(
            config          = account_config,
            list_files      = list_files,
            adjustment_path = ADJUSTMENT_PATH,
            save_path       = SAVE_PATH,
            period_filter   = PERIOD_FILTER,
            # Pass a list to skip the interactive prompt, e.g.:
            # exclude_adjustment_types=["AMS Commission Fee"],
            # auto_continue=True,   # skip the missing-order pause
        )
    print("\n\nAll SHP accounts processed successfully.")

23:01:40  INFO      ============================================================
23:01:40  INFO      Processing: vinda SHP
23:01:40  INFO      ============================================================
C:\Users\thanh.bui\Documents\App_download\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\thanh.bui\Documents\App_download\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
23:01:56  INFO      Adjustment types found:
['Adjustment/Compensation for Return of Goods/Funds']


Enter types to EXCLUDE (comma-separated), or press Enter to keep all:  


23:02:19  INFO      ✓ No missing orders.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Linked Order No.   1 non-null      object 
 1   classify           1 non-null      object 
 2   Adjustment Amount  1 non-null      float64
dtypes: float64(1), object(2)
memory usage: 156.0+ bytes
None


23:02:24  INFO      ✓ Saved → C:\Users\thanh.bui\Documents\CLAIMBACK\ID\Retail\vinda\vinda_SHP.xlsx




All SHP accounts processed successfully.


## LAZADA

In [6]:
# ─────────────────────────────────────────────
# LAZADA ACCOUNT CONFIGS
# ─────────────────────────────────────────────
# FIX: LZD_ACCOUNTS now drives the loop just like ACCOUNTS for SHP.
#      Added "brand" key per account (used for master mapping filter).
# ─────────────────────────────────────────────
LZD_ACCOUNTS = [
    {
        "label":       f"{ACCOUNT_NAME}_LZD",
        "output_name": f"{ACCOUNT_NAME}_LZD.xlsx",
        "get_data":    ["lzd", "income"],
        "remove":      [],
        "brand":       ACCOUNT_NAME,
    }
]

# ═══════════════════════════════════════════════════════════════════
# LZD ORDER REPORT
# FIX 1: Removed all implicit outer-scope reads — income_list, lzd,
#         beta_inc, get_data, remove were never passed in before.
# FIX 2: Built a proper file-loading loop so df_lzd is always defined.
# FIX 3: Renamed inner "remove" list → "revenue_rows" to stop it
#         from shadowing the file-filter parameter.
# FIX 4: Replaced undefined "beta_inc" with df_lzd (full raw frame)
#         when appending lost_order rows.
# FIX 5: Master join now uses correctly-cased "Fee name" column.
# ═══════════════════════════════════════════════════════════════════
def lzd_order_report(list_files, master, get_data, remove_keywords, brand):
    frames = []
    for filename in list_files:
        s = filename.lower()
        if "~$" in filename:
            continue
        if all(k in s for k in get_data) and not any(k in s for k in remove_keywords):
            df = pd.read_excel(filename, dtype={"Order Number": str, "Order Line ID": str})
            frames.append(df)

    if not frames:
        raise FileNotFoundError(
            f"No LZD file matched keywords {get_data} (excluding {remove_keywords}). "
            "Check that the raw-data folder contains the right files."
        )

    df_lzd = pd.concat(frames, ignore_index=True)

    # ── Master mapping ──────────────────────────────────────────
    mas  = pd.read_csv(master)
    filt = (mas["Venture"] == "ID") & (mas["Brand"].str.lower() == brand) & (mas["Platform"] == "lazada")
    mas  = mas[filt].copy()
    mas.columns  = mas.columns.str.strip()
    mas["Fee name"] = mas["Fee name"].str.strip().str.title()

    master_discount = mas[mas["NMV included?"] == "Yes"]

    # create the fees to merge with final data
    master_fee = mas[mas["NMV included?"] != "Yes"]['Fee name']
    fee_cols = df_lzd[df_lzd["Fee Name"].isin(master_fee)]['Fee Name'].unique().tolist()
    to_map = df_lzd.pivot_table(index='Order Line ID', columns='Fee Name', values='Amount(Include Tax)', aggfunc='sum', fill_value=0).reset_index()

    to_map.columns.name = None                 # clean up column index name from pivot
    to_map = to_map[['Order Line ID'] + fee_cols].reset_index()
        
    # ── Revenue rows (renamed from "remove" to avoid shadowing) 
    revenue_rows    = ["Item Price Credit", "Reversal Item Price"]
    df_lzd['Fee Name'] = df_lzd['Fee Name'].str.strip().str.title()
    inc             = df_lzd[df_lzd["Fee Name"].isin(revenue_rows)].copy()

    # ── Discount / voucher rows ──────────────────────────────────
    filter_discount = master_discount["Fee name"].unique()
    discount        = df_lzd[
        df_lzd["Fee Name"].isin(filter_discount) & (~df_lzd["Fee Name"].isin(revenue_rows))
    ]
    gr_discount = (
        discount.groupby("Order Line ID")["Amount(Include Tax)"]
        .sum().rename("discount").reset_index()
    )

    tt_amount = df_lzd[
        df_lzd["Fee Name"].isin(filter_discount) & df_lzd["Fee Name"].isin(revenue_rows)
    ]
    tt_amount = (
        tt_amount.groupby("Order Line ID")["Amount(Include Tax)"]
        .sum().rename("Total amount").reset_index()
    )

    # ── Append orders with discount but no income row ────────────
    lost_order = gr_discount[
        ~gr_discount["Order Line ID"].isin(inc["Order Line ID"].unique())
    ]["Order Line ID"].unique()

    inc_append = df_lzd[df_lzd["Order Line ID"].isin(lost_order)].copy()  # FIX: was undefined "beta_inc"
    inc        = pd.concat([inc, inc_append], ignore_index=True)
    inc        = inc.drop_duplicates(subset=["Order Line ID"], keep="first")

    # ── Build final columns ──────────────────────────────────────
    inc = inc.merge(tt_amount,   how="left", on="Order Line ID")
    inc["Amount(Include Tax)"] = inc["Total amount"]
    inc = inc.merge(gr_discount, how="left", on="Order Line ID")
    inc["discount"] = inc["discount"].fillna(0)

    inc["TNMV"] = np.where(
        inc["Order Line ID"].isin(lost_order),
        inc["discount"],
        inc["discount"] + inc["Amount(Include Tax)"],
    )

    order = inc.drop(columns={"Fee Name"})

    # ── Flag fee names not yet in master ─────────────────────────
    new_fee = df_lzd[~df_lzd["Fee Name"].isin(mas["Fee name"])]["Fee Name"].unique()
    return order, new_fee, to_map

def format_df(final, to_map):
    final = final.merge(to_map, how ='left', on = 'Order Line ID')
    final = final.drop(columns=["index"], errors="ignore")
    return final

# ═══════════════════════════════════════════════════════════════════
# LZD PIPELINE RUNNER  –  mirrors run_pipeline() for SHP
# ═══════════════════════════════════════════════════════════════════
def run_lzd_pipeline(config, list_files, master, save_path):
    print(f"\n{'='*60}")
    print(f"  Processing: {config['label']}")
    print(f"{'='*60}")

    order, new_fee, to_map = lzd_order_report(
        list_files,
        master,
        config["get_data"],
        config["remove"],
        config["brand"],
    )

    if len(new_fee) > 0:
        print("\n⚠ Fee names not in master mapping (review before saving):")
        print(new_fee)
        print("Press Enter to continue anyway (or Ctrl-C to abort):")
        input()

    final = format_df(order, to_map)
    
    output_path = os.path.join(save_path, config["output_name"])
    final.to_excel(output_path, index=False)
    print(f"\n✓ Saved → {output_path}")
    return order

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MAIN  –  run all LZD accounts in sequence
# ═══════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    os.chdir(RAW_DATA_DIR)
    list_files = os.listdir()

    lzd_results = {}
    for cfg in LZD_ACCOUNTS:
        lzd_results[cfg["label"]] = run_lzd_pipeline(
            cfg, list_files, MASTER, SAVE_PATH
        )

    print("\n\nAll LZD accounts processed successfully.")

## TIKTOK

In [8]:
# ═══════════════════════════════════════════════════════════════════
# TIKTOK HELPER FUNCTIONS
# NOTE: These use TTS column names (Seller SKU, SKU Subtotal Before
#       Discount) vs SHP (SKU Reference No., Total Buyer Payment).
#       Prefixed with tts_ to avoid name collisions with SHP versions.
# ═══════════════════════════════════════════════════════════════════
def tts_dp_refund_distribution(df_ref, refund_target, tol=0.01):
    skus, prices = [], []
    for _, row in df_ref.iterrows():
        for _ in range(int(row["total_quant"])):
            skus.append(row["Seller SKU"])
            prices.append(row["unit_price"])

    prices = np.array(prices)
    n      = len(prices)
    result = []

    def dfs(i, current_sum, picked):
        if abs(current_sum - refund_target) <= tol:
            result.extend(picked)
            return True
        if i == n or current_sum > refund_target + tol:
            return False
        if dfs(i + 1, current_sum + prices[i], picked + [skus[i]]):
            return True
        if dfs(i + 1, current_sum, picked):
            return True
        return False

    found = dfs(0, 0.0, [])
    return dict(Counter(result)) if found else None

def tts_fallback_refund_distribution(df_ref, refund_target):
    skus        = df_ref["Seller SKU"].tolist()
    unit_prices = df_ref["unit_price"].to_numpy()
    max_qty     = df_ref["total_quant"].astype(int).to_numpy()
    n           = len(skus)
    refund_qty  = np.zeros(n, dtype=int)
    current_sum = 0.0

    while True:
        best_idx, best_gap = None, abs(refund_target - current_sum)
        for i in range(n):
            if refund_qty[i] >= max_qty[i]:
                continue
            gap = abs(refund_target - (current_sum + unit_prices[i]))
            if gap < best_gap:
                best_gap, best_idx = gap, i
        if best_idx is None:
            break
        refund_qty[best_idx] += 1
        current_sum += unit_prices[best_idx]

    return dict(zip(skus, refund_qty)), current_sum


def tts_resolve_partial_refund(df_order_one, refund_target, tol=0.01):
    cons = (
        (df_order_one["Order Status"] == "Canceled") |
        (df_order_one["Cancelation/Return Type"] == "Cancel") |
        (df_order_one["Cancelation/Return Type"] == "Return/Refund")
    )
    df_ref = df_order_one[cons].copy()

    if df_ref.empty or refund_target <= 0:
        return {"method": "NO_DATA", "breakdown": None, "refunded_amount": 0.0}

    df_ref["unit_price"] = df_ref["SKU Subtotal Before Discount"] / df_ref["total_quant"]
    dp_result = tts_dp_refund_distribution(df_ref, refund_target, tol)

    if dp_result is not None:
        refunded_amount = sum(
            dp_result[sku] * df_ref.loc[df_ref["Seller SKU"] == sku, "unit_price"].iloc[0]
            for sku in dp_result
        )
        return {"method": "DP_EXACT", "breakdown": dp_result, "refunded_amount": refunded_amount}

    fallback_result, approx_sum = tts_fallback_refund_distribution(df_ref, refund_target)
    return {"method": "GREEDY_ESTIMATE", "breakdown": fallback_result, "refunded_amount": approx_sum}


def tts_process_all_orders(df_order, df_income):
    results = []
    for order_id, group in df_order.groupby("Order ID"):
        row = df_income[df_income["Order/Adjustment ID"] == order_id]
        if row.empty:
            continue
        refund_target = float(row["total_refund"].iloc[0])
        res           = tts_resolve_partial_refund(group, refund_target)
        results.append({
            "Order ID":         order_id,
            "Refund target":    refund_target,
            "Refund method":    res["method"],
            "Refund breakdown": res["breakdown"],
            "Refunded amount":  res["refunded_amount"],
            "Variance":         res["refunded_amount"] - refund_target,
        })
    return pd.DataFrame(results)


def tts_expand_refund_breakdown(result_df):
    rows = []
    for _, r in result_df.iterrows():
        if not isinstance(r["Refund breakdown"], dict):
            continue
        for sku, qty in r["Refund breakdown"].items():
            rows.append({"Order ID": r["Order ID"], "Seller SKU": sku, "correct_refund_quantity": int(qty)})
    return pd.DataFrame(rows)


def tts_apply_correct_refund_quantity(df_order, result_df):
    refund_map = tts_expand_refund_breakdown(result_df)
    df_out     = df_order.merge(refund_map, on=["Order ID", "Seller SKU"], how="left")
    df_out["correct_refund_quantity"] = df_out["correct_refund_quantity"].fillna(0).astype(int)
    return df_out


def tts_prepare_working_file(df_inc, df_orders, inc_ord):
    numeric_cols     = df_inc.select_dtypes(include="number").columns
    non_numeric_cols = df_inc.select_dtypes(exclude="number").columns.difference(
        ["Order/Adjustment ID", "Related order ID"]
    )
    agg_dict = {col: "sum"   for col in numeric_cols}
    agg_dict.update({col: "first" for col in non_numeric_cols})
    reim_list = df_inc[df_inc['Transaction type'] != "Order"]['Related order ID'].drop_duplicates()
    inc_grp = df_inc.groupby('Related order ID')['Total settlement amount'].sum().reset_index()
    to_map = df_inc.groupby("Related order ID", as_index=False).agg(agg_dict)
    df_inc = df_inc.groupby("Order/Adjustment ID", as_index=False).agg(agg_dict)

    order   = inc_ord["Related order ID"].drop_duplicates()
    
    df_comp = df_orders[df_orders["Order ID"].isin(order)].copy()

    refund_list = df_inc[df_inc["Refund subtotal before seller discounts"] < 0]["Order/Adjustment ID"]

    prv_ord = df_inc[df_inc["Total Revenue"] < 0]["Order/Adjustment ID"]
    
    df_comp["Order Status"]            = np.where(~df_comp["Order ID"].isin(refund_list), "Completed",     df_comp["Order Status"])
    df_comp["Order Substatus"]         = np.where(~df_comp["Order ID"].isin(refund_list), "Completed",     df_comp["Order Substatus"])
    df_comp["Cancelation/Return Type"] = np.where(~df_comp["Order ID"].isin(refund_list), np.nan,          df_comp["Cancelation/Return Type"])
    df_comp["return_ord"]              = np.where( df_comp["Order ID"].isin(refund_list), "yes",           "no")
    df_comp["Cancelation/Return Type"] = np.where( df_comp["Order ID"].isin(refund_list), "Return/Refund", df_comp["Cancelation/Return Type"])

    
    df_comp["quantity mapping"] = df_comp["Seller SKU"].str.extract(r'[xX](\d+)$')
    df_comp["quantity mapping"] = pd.to_numeric(df_comp["quantity mapping"], errors="coerce").fillna(1).astype(int)
    df_comp['sku_mapping']      = np.where(df_comp['Seller SKU'].str.contains('GIFT'), 
                                   df_comp['Seller SKU'], 
                                   df_comp['Seller SKU'].str.replace(r'[xX]\d+$', '', regex=True))
    
    # Now safely replace all NaN (from missing "x" OR non-numeric strings) with 1
    df_comp["quantity mapping"] = df_comp["quantity mapping"].fillna(1).astype("int")
    df_comp["total_quant"]      = df_comp["Quantity"] * df_comp["quantity mapping"]

    return df_comp, refund_list, df_inc, prv_ord , reim_list, inc_grp, to_map


def tts_partial_table(df_comp, df_inc, refund_list):
    df_partial = df_comp[df_comp["Order ID"].isin(refund_list)].copy()

    refund_amt = df_inc[df_inc["Order/Adjustment ID"].isin(refund_list)]
    refund_amt = (
        refund_amt.groupby("Order/Adjustment ID")["Refund subtotal before seller discounts"]
        .sum().rename("total_refund").reset_index()
    )
    refund_amt["total_refund"] = refund_amt["total_refund"] * -1
    return refund_amt, df_partial


def tts_get_compensation(append_df, df_inc, df_comp, refund_list, prv_ord, reim_list, inc_grp, mp):
    # FIX: Added .copy() to silence SettingWithCopyWarning
    df_final = df_comp[~df_comp["Order ID"].isin(refund_list)].copy()
    df_final["correct_refund_quantity"] = 0
    df_final = pd.concat([df_final, append_df], ignore_index=True)
    df_final["Selling quantity"] = df_final["total_quant"] - df_final["correct_refund_quantity"]

    df_seller = append_df.groupby("Order ID")["correct_refund_quantity"].sum().reset_index()
    df_seller = df_seller.merge(
        df_inc[["Order/Adjustment ID", "Refund of seller discounts", "Refund subtotal before seller discounts"]],
        how="left", left_on="Order ID", right_on="Order/Adjustment ID",
    )
    df_seller["rf_seller_per_unit"] = (df_seller["Refund of seller discounts"]             / df_seller["correct_refund_quantity"]).round(2)
    df_seller["rf_amt"]             = (df_seller["Refund subtotal before seller discounts"] / df_seller["correct_refund_quantity"]).round(2)

    df_final = df_final.merge(df_seller[["Order ID", "rf_seller_per_unit", "rf_amt"]], how="left", on="Order ID")
    df_final["rf_seller_per_unit"] = df_final["rf_seller_per_unit"].fillna(0)
    df_final["rf_amt"]             = df_final["rf_amt"].fillna(0)

    df_final["Refund Seller Discount"] = (df_final["correct_refund_quantity"] * df_final["rf_seller_per_unit"]).round(2)
    df_final["Refund Amount"]          = (df_final["SKU Subtotal Before Discount"] / df_final["total_quant"] * df_final["correct_refund_quantity"]).round(2)
    df_final["Discounted price"]       = (
        df_final["SKU Subtotal Before Discount"] - df_final["SKU Seller Discount"] + df_final["Refund Seller Discount"]
    ).round(2)
    df_final["TNMV"] = np.where(
        df_final["Selling quantity"] > 0,
        (df_final["Discounted price"] - df_final["Refund Amount"]).round(2),
        0,
    )

    df_final['Selling quantity'] = np.where(df_final["Order ID"].isin(prv_ord), -df_final["correct_refund_quantity"], df_final['Selling quantity'])
    df_final["TNMV"] = np.where(df_final["Order ID"].isin(prv_ord) , -df_final["Refund Amount"] + df_final["Refund Seller Discount"], df_final["TNMV"])
    
    df_final = df_final.merge(mp[['SKU' , 'Purchase Price (Including VAT)']], how='left', left_on= "sku_mapping", right_on = "SKU")
    
    df_final['Total Purchase price'] = df_final['Purchase Price (Including VAT)'].fillna(0) * df_final['Selling quantity'] 

    
    # correct reimbursement orders
    reim = df_final.groupby('Order ID')['total_quant'].sum().reset_index()
    reim = reim[reim['Order ID'].isin(reim_list)]
    reim = reim.merge(inc_grp, how = 'left', left_on = 'Order ID', right_on = 'Related order ID')
    reim['reim_per_quant'] = reim['Total settlement amount'] / reim['total_quant']
    df_final = df_final.merge(reim[['Order ID', 'reim_per_quant']], how='left', on = 'Order ID')
    df_final['TNMV'] = np.where(df_final['reim_per_quant'].notna(), df_final['reim_per_quant'] * df_final['total_quant'], df_final['TNMV'])
    
    df_final = df_final.drop(columns={"rf_seller_per_unit", "rf_amt", "return_ord", 'reim_per_quant', 'Purchase Price (Including VAT)', "SKU"})
    return df_final

def format_df(df_final, inc, fees):
    # inc_df here is the "to_map" dataframe
    cols = [c for c in fees if c in inc.columns]

    df_final['rate'] = (df_final['total_quant'] / df_final.groupby('Order ID')['total_quant'].transform('sum'))
    final = df_final.merge(inc[['Related order ID'] + cols], how = 'left', left_on = 'Order ID', right_on= 'Related order ID')
    final[cols] = final[cols].fillna(0)
    final[cols] = final[cols].mul(final['rate'], axis = 0)

    # extract only column we want to present
    final = final[['Order ID', 'Order Status', 'Seller SKU', 'Product Name', 'Variation', 'Created Time', 'Paid Time', 'Purchase Channel', 'sku_mapping', 'total_quant', 'Selling quantity',
                   'TNMV', 'Total Purchase price', 'rate'] + cols]

    return final

In [9]:
# ─────────────────────────────────────────────
# TIKTOK ACCOUNT CONFIGS
# ─────────────────────────────────────────────
# income_keywords and order_keywords replace the previous hardcoded
# "tts" + "income"/"order" check so each brand's files are matched
# independently — same pattern as ACCOUNTS for SHP.
# ─────────────────────────────────────────────
TTS_ACCOUNTS = [
    {
        "label":           f"{ACCOUNT_NAME}_TTS",
        "output_name":     f"{ACCOUNT_NAME}_TTS.xlsx",
        "income_keywords": ["tts", "income"],
        "order_keywords":  ["tts", "order"],
        "remove":          [],
    }
]


def tts_load_files(list_files, income_keywords, order_keywords, remove):
    """Load TTS income (.xlsx) and order (.csv) files matching the keyword config."""
    inc_frames, order_frames = [], []
    for filename in list_files:
        s = filename.lower()
        if "~$" in s:
            continue
        matches_income = all(k in s for k in income_keywords) and not any(k in s for k in remove)
        matches_order  = all(k in s for k in order_keywords)  and not any(k in s for k in remove)

        if matches_income:
            excel = pd.read_excel(
                filename, sheet_name="Order details",
                dtype={"Related order ID": str, "Order/Adjustment ID": str},
            )
            excel.columns = excel.columns.str.strip()
            inc_frames.append(excel)

            summary = pd.read_excel(filename, sheet_name="Reports", skiprows = 5)
            
        elif matches_order:
            excel = pd.read_csv(filename, dtype={"Order ID": str})
            order_frames.append(excel)

    if not inc_frames:
        raise FileNotFoundError(f"No TTS income file matched {income_keywords} (excluding {remove}).")
    if not order_frames:
        raise FileNotFoundError(f"No TTS order file matched {order_keywords} (excluding {remove}).")
    
    return pd.concat(inc_frames, ignore_index=True), pd.concat(order_frames, ignore_index=True), summary

def get_fee_name(summary: pd.DataFrame, income: pd.DataFrame):
    summary = summary.copy()
    summary.columns = ["no1", "no2", "fee", "sub-fee", "sub-fee_2", "amount"]

    summary["fee"] = summary["fee"].ffill()
    # summary["sub-fee"] = summary["sub-fee"].where(
    #     summary["sub-fee"].notna(),
    #     summary["sub-fee_2"]
    # )

    fee_rows = summary[
        (summary["fee"] == "Total Fees") &
        summary["sub-fee"].notna() &
        (summary["amount"] != 0) 
        # & (summary["sub-fee"] != "Shipping cost")
    ]

    conds = ["Logistics reimbursement"]
    adjust = summary[
        (summary["fee"] == "Adjustments") &
        (~summary["sub-fee"].isin(conds))
    ]

    fee_rows = pd.concat([fee_rows, adjust], ignore_index=True)
    fee_row = fee_rows['sub-fee']
    
    fee_cols = [
        c for c in fee_rows["sub-fee"].dropna().unique().tolist()
        if c in income.columns
    ]

    # Pivot Transaction type into columns
    id_cols = [
        c for c in income.columns
        if c not in ["Transaction type", "Total settlement amount"]
    ]

    df_pivot = (
        income.pivot_table(
            index=id_cols,
            columns="Transaction type",
            values="Total settlement amount",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
    )
    df_pivot.columns.name = None

    # Unpivot fee columns separately
    pivot = income.melt(
        id_vars=["Order source"],
        value_vars=fee_cols,
        var_name="Fee Name",
        value_name="Amount (+VAT)"
    )
    pivot = pivot.groupby(["Fee Name", "Order source"])["Amount (+VAT)"].sum().reset_index()
    return pivot, fee_row
# ═══════════════════════════════════════════════════════════════════
# TTS PIPELINE RUNNER  –  one call per TTS account config
# ═══════════════════════════════════════════════════════════════════
def run_tts_pipeline(config, list_files, save_path):
    print(f"\n{'='*60}")
    print(f"  Processing: {config['label']}")
    print(f"{'='*60}")

    inc_tts, order_tts, summary = tts_load_files(
        list_files,
        config["income_keywords"],
        config["order_keywords"],
        config["remove"],
    )

    # ── Strip whitespace from key ID columns ────────────────────
    id_cols = ["Order/Adjustment ID", "Related order ID"]
    for i in id_cols:
        inc_tts[i] = inc_tts[i].astype("string").str.strip()
    order_tts["Order ID"] = order_tts["Order ID"].str.strip()

    # ── Type exclusion prompt ────────────────────────────────────
    print(f"\nType values found:\n{inc_tts['Transaction type'].unique()}")
    print("Enter Types to EXCLUDE (comma-separated), or press Enter to keep all:")
    raw           = input().strip()
    exclude_types = [x.strip() for x in raw.split(",")] if raw else []
    inc_tts       = inc_tts[~inc_tts["Transaction type"].isin(exclude_types)]

    # ── Missing order check ──────────────────────────────────────
    order_tts["index"] = order_tts.groupby("Order ID").cumcount() + 1
    inc_ord     = inc_tts.drop_duplicates(subset=["Related order ID"])
    order_ord   = order_tts["Order ID"].drop_duplicates()
    missing_ord = inc_ord[~inc_ord["Related order ID"].isin(order_ord)]

    if missing_ord.empty:
        print("\n✓ No missing orders.")
    else:
        print("\n⚠ Missing orders:")
        print(missing_ord[["Order/Adjustment ID", "Related order ID", "Order created time", "Order settled time"]])

    print("Press Enter to continue (or type 'stop' to abort):")
    text = input().strip()
    if text.lower() == "stop":
        print("Stopped. Download missing data and re-run.")
        return None

    # ── Pipeline ─────────────────────────────────────────────────
    tts_comp, tts_refund_list, tts_income, prv_order, reim_list, inc_grp, to_map = tts_prepare_working_file(inc_tts, order_tts, inc_ord)
    refund_amt, tts_partial                                              = tts_partial_table(tts_comp, tts_income, tts_refund_list)
    check                                                                = tts_process_all_orders(tts_partial, refund_amt)
    tts_append                                                           = tts_apply_correct_refund_quantity(tts_partial, check)
    final_df                                                             = tts_get_compensation(tts_append, tts_income, tts_comp, tts_refund_list, prv_order, reim_list, inc_grp, mp)
    summary_pivot, fee_rows                                              = get_fee_name(summary, tts_income)
    tts_final                                                            = format_df(final_df, to_map, fee_rows)
    
    output_path = os.path.join(save_path, config["output_name"])
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        tts_final.to_excel(writer, sheet_name="TTS_order" ,index=False)
        summary_pivot.to_excel(writer, sheet_name="pivot", index=False)
    print(f"\n✓ Saved → {output_path}")
    return tts_final

In [10]:
# ═══════════════════════════════════════════════════════════════════
# MAIN  –  run all TTS accounts in sequence
# ═══════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    os.chdir(RAW_DATA_DIR)
    list_files = os.listdir()

    tts_results = {}
    for cfg in TTS_ACCOUNTS:
        tts_results[cfg["label"]] = run_tts_pipeline(cfg, list_files, SAVE_PATH)

    print("\n\nAll TTS accounts processed successfully.")


  Processing: vinda_TTS

Type values found:
['Order']
Enter Types to EXCLUDE (comma-separated), or press Enter to keep all:



✓ No missing orders.
Press Enter to continue (or type 'stop' to abort):



✓ Saved → C:\Users\thanh.bui\Documents\CLAIMBACK\ID\Retail\vinda\vinda_TTS.xlsx


All TTS accounts processed successfully.
